# W12 · 01 포틀란다이트 + 물 ReaxFF 2,000스텝

Claude Code: 「@notebooks/01_reaxff_portlandite.ipynb 를 실행하고 결과를 요약해」 · `python ../run_nb.py notebooks/01_reaxff_portlandite.ipynb` → `out/reaxff_summary.json`. LAMMPS(REAXFF) · Packmol = mdai. (선택) Colab 이면 첫 셀의 설치 주석을 푼다(condacolab · 수 분). 설명은 덱 12~14장 · `../README.md`.

In [ ]:
# !pip -q install condacolab && python -c 'import condacolab; condacolab.install()'   # (선택) Colab 1회 · 재시작 뒤 아래 · 로컬 mdai 은 건너뜀
# !conda install -y -c conda-forge lammps packmol
import subprocess, json, os, shutil
LMP = shutil.which('lmp') or shutil.which('lmp_serial') or 'lmp'
PACKMOL = shutil.which('packmol') or 'packmol'
print(LMP, PACKMOL)

In [ ]:
STEPS = 2000   # 바꿔 보기: 8000 (2 ps)
r = subprocess.run(['python3', 'scripts/reaxff_input.py', '--packmol', PACKMOL, '--lmp', LMP, '--steps', str(STEPS)], cwd='..', capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-500:])

In [ ]:
s = json.load(open('../out/reaxff_summary.json'))
print('원자', s['n_atoms'], '· 물', s['n_water'], '· 스텝', s['steps'], '· 벽시계', s.get('loop_time_s'), 's')
for f in s.get('frames', []): print(f)
for c in s.get('charges', []): print(c)

In [ ]:
# 온도 · 퍼텐셜 에너지 시계열 (matplotlib 있으면 그림)
try:
    import matplotlib.pyplot as plt
    th = s['thermo']; st = [t['step'] for t in th]
    fig, ax = plt.subplots(1, 2, figsize=(10, 3))
    ax[0].plot(st, [t['temp'] for t in th]); ax[0].set_title('T (K)')
    ax[1].plot(st, [t['pe'] for t in th]); ax[1].set_title('PE (kcal/mol)')
    plt.tight_layout(); plt.savefig('../out/reaxff_thermo.png', dpi=120); print('saved out/reaxff_thermo.png')
except Exception as e: print('그림 생략', e)

## 확인
- `frames` 마지막 행: Ca–O_w 결합 수(BO ≥ 0.3) · 물 O 의 H 개수 분포 — H 2개만 있으면 해리 없음
- `charges`: Ca · O · H 전하가 스텝마다 변한다(QEq) — 고전 힘장의 고정 전하와 다른 점
- Claude Code 지시 P12-2 가 이 결과를 읽는다 → `out/P12-2_result.md`